#### Dealing With Text Data

##### Encoding Text

In [18]:
import pandas as pd 
data = pd.read_csv("./datasets/inaugural_speeches.csv")

In [19]:
# print first five columns of the text column
data["text"].head()

0    Fellow-Citizens of the Senate and of the House...
1    Fellow Citizens:  I AM again called upon by th...
2    WHEN it was first perceived, in early times, t...
3    Friends and Fellow-Citizens:  CALLED upon to u...
4    PROCEEDING, fellow-citizens, to that qualifica...
Name: text, dtype: object

In [20]:
# replace all non letter characters with white space
data["text_clean"] = data["text"].str.replace("[^a-zA-Z]", " ", regex=True)

# change the text column to lower case
data["text_clean"] = data["text_clean"].str.lower()

# display first five rows
display(data["text_clean"].head())

0    fellow citizens of the senate and of the house...
1    fellow citizens   i am again called upon by th...
2    when it was first perceived  in early times  t...
3    friends and fellow citizens   called upon to u...
4    proceeding  fellow citizens  to that qualifica...
Name: text_clean, dtype: object

High Level Text Features

In [21]:
# find the length of each text
data["char_cnt"] = data["text_clean"].str.len()

# find the number of words in each text
data["word_cnt"] = data["text_clean"].str.split().str.len()

# find average lenth of each word in a text
data["avg_word_length"] = data["char_cnt"] / data["word_cnt"]

# display first five rows
display(data[["text_clean", "char_cnt", "word_cnt", "avg_word_length"]].head())

,text_clean,char_cnt,word_cnt,avg_word_length
0,fellow citizens of the senate and of the house...,8616,1432,6.016760
1,fellow citizens i am again called upon by th...,787,135,5.829630
2,when it was first perceived in early times t...,13871,2323,5.971158
3,friends and fellow citizens called upon to u...,10144,1736,5.843318
4,proceeding fellow citizens to that qualifica...,12902,2169,5.948363


##### Word Counts

Counting Words (1)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
cv.fit(data["text_clean"])
print(cv.get_feature_names_out())

['abandon' 'abandoned' 'abandonment' ... 'zealous' 'zealously' 'zone']


Counting Words (2)

In [26]:
cv_transformed = cv.transform(data["text_clean"])
print(cv_transformed.toarray().shape)
print(cv_transformed.toarray())

(58, 9043)
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 ...
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Limiting Your Features

In [27]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(min_df=0.2, max_df=0.8)
cv.fit(data["text_clean"])
cv_transformed = cv.transform(data["text_clean"])
cv_array = cv_transformed.toarray()
print(cv_array.shape)

(58, 818)


Text to a DataFrame

In [32]:
cv_df = pd.DataFrame(cv_array, columns=cv.get_feature_names_out()).add_prefix("Count_")
data = pd.concat([data, cv_df], axis=1)
display(data.columns)

Index(['Name', 'Inaugural Address', 'Date', 'text', 'text_clean', 'char_cnt',
       'word_cnt', 'avg_word_length', 'Count_abiding', 'Count_ability',
       ...
       'Count_women', 'Count_words', 'Count_work', 'Count_wrong', 'Count_year',
       'Count_years', 'Count_yet', 'Count_you', 'Count_young', 'Count_your'],
      dtype='object', length=826)

#### Term Frequency - Inverse Document Frequency

Tf-idf

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(max_features=100, stop_words="english")
tv_transformed = tv.fit_transform(data["text_clean"])

tv_df = pd.DataFrame(tv_transformed.toarray(), columns=tv.get_feature_names_out()).add_prefix("TFIDF_")
display(tv_df.head())

,TFIDF_action,TFIDF_administration,TFIDF_america,TFIDF_american,TFIDF_americans,TFIDF_believe,TFIDF_best,TFIDF_better,TFIDF_change,TFIDF_citizens,...,TFIDF_things,TFIDF_time,TFIDF_today,TFIDF_union,TFIDF_united,TFIDF_war,TFIDF_way,TFIDF_work,TFIDF_world,TFIDF_years
0,0.000000,0.133415,0.000000,0.105388,0.0,0.000000,0.000000,0.000000,0.000000,0.229644,...,0.000000,0.045929,0.0,0.136012,0.203593,0.000000,0.060755,0.000000,0.045929,0.052694
1,0.000000,0.261016,0.266097,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.179712,...,0.000000,0.000000,0.0,0.000000,0.199157,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.092436,0.157058,0.073018,0.0,0.000000,0.026112,0.060460,0.000000,0.106072,...,0.032030,0.021214,0.0,0.062823,0.070529,0.024339,0.000000,0.000000,0.063643,0.073018
3,0.000000,0.092693,0.000000,0.000000,0.0,0.090942,0.117831,0.045471,0.053335,0.223369,...,0.048179,0.000000,0.0,0.094497,0.000000,0.036610,0.000000,0.039277,0.095729,0.000000
4,0.041334,0.039761,0.000000,0.031408,0.0,0.000000,0.067393,0.039011,0.091514,0.273760,...,0.082667,0.164256,0.0,0.121605,0.030338,0.094225,0.000000,0.000000,0.054752,0.062817


Inspecting tfidf values

In [35]:
tv_df.iloc[0].sort_values(ascending=False)

TFIDF_government    0.367430
TFIDF_public        0.333237
TFIDF_present       0.315182
TFIDF_duty          0.238637
TFIDF_citizens      0.229644
                      ...   
TFIDF_state         0.000000
TFIDF_spirit        0.000000
TFIDF_today         0.000000
TFIDF_war           0.000000
TFIDF_work          0.000000
Name: 0, Length: 100, dtype: float64

Fitting on Seen data and Transforming on Unseen data

In [43]:
n_train = int(data.shape[0] * (0.8))

data_train = data.iloc[:n_train, :]
data_test = data.iloc[n_train+1:, :]

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(max_features=100, stop_words="english")
tv.fit_transform(data_train["text_clean"])
test_tv_transformed = tv.transform(data_test["text_clean"])

test_tv_df = pd.DataFrame(test_tv_transformed.toarray(), columns=tv.get_feature_names_out()).add_prefix("TFIDF_")

display(test_tv_df.head())

,TFIDF_action,TFIDF_administration,TFIDF_america,TFIDF_american,TFIDF_best,TFIDF_business,TFIDF_citizens,TFIDF_commerce,TFIDF_common,TFIDF_confidence,...,TFIDF_subject,TFIDF_support,TFIDF_time,TFIDF_union,TFIDF_united,TFIDF_war,TFIDF_way,TFIDF_work,TFIDF_world,TFIDF_years
0,0.000000,0.00000,0.123606,0.132355,0.132355,0.000000,0.000000,0.00000,0.047197,0.042213,...,0.0,0.000000,0.074152,0.000000,0.079092,0.042213,0.053059,0.096579,0.222456,0.043152
1,0.036682,0.06526,0.256366,0.030501,0.061003,0.049356,0.075271,0.00000,0.000000,0.000000,...,0.0,0.097889,0.205061,0.000000,0.054680,0.029184,0.036682,0.233697,0.230694,0.059667
2,0.000000,0.00000,0.213566,0.152455,0.087117,0.000000,0.107494,0.00000,0.023299,0.020839,...,0.0,0.023299,0.183028,0.130964,0.039044,0.020839,0.078579,0.119193,0.292845,0.149117
3,0.066046,0.00000,0.269257,0.054917,0.027459,0.000000,0.045175,0.00000,0.000000,0.000000,...,0.0,0.000000,0.161530,0.033023,0.073839,0.078819,0.099070,0.210385,0.230756,0.053715
4,0.000000,0.00000,0.656045,0.104072,0.026018,0.042101,0.064207,0.03129,0.000000,0.024894,...,0.0,0.000000,0.153054,0.000000,0.000000,0.049789,0.093871,0.170867,0.437296,0.000000


##### N Grams

Using Longer N Grams

In [45]:
from sklearn.feature_extraction.text import CountVectorizer

cv_trigram_vec = CountVectorizer(max_features=100, stop_words="english", ngram_range=(3, 3))
cv_trigram = cv_trigram_vec.fit_transform(data["text_clean"])

print(cv_trigram_vec.get_feature_names_out())

['ability preserve protect' 'agriculture commerce manufactures'
 'america celebrate th' 'america ideal freedom' 'america stands world'
 'america work preserve' 'american merchant marine'
 'americans fellow citizens' 'amity mutual concession' 'anchor peace home'
 'angel rides whirlwind' 'antirepublican tendencies preservation'
 'appear fellow citizens' 'ask bow heads' 'ask just government'
 'aspirations great people' 'assigned executive branch'
 'best ability preserve' 'best interests country' 'bless god bless'
 'bless united states' 'chief justice mr' 'children children children'
 'citizens united states' 'civil religious liberty' 'civil service reform'
 'commerce united states' 'confidence fellow citizens'
 'congress extraordinary session' 'constitution does expressly'
 'constitution united states' 'coordinate branches government'
 'day task people' 'defend constitution united'
 'distinguished guests fellow' 'does expressly say' 'equal exact justice'
 'era good feeling' 'executive bra

Finding The Most Common Words

In [47]:
cv_tri_df = pd.DataFrame(cv_trigram.toarray(), columns=cv_trigram_vec.get_feature_names_out())\
    .add_prefix("Count_")



In [50]:
cv_tri_df.sum().sort_values(ascending=False).head()

Count_constitution united states    20
Count_people united states          13
Count_mr chief justice              10
Count_preserve protect defend       10
Count_president united states        8
dtype: int64